# **End-to-End LLM Prototype for Automated Car Review Analytics & QA :**

![image](car.jpeg)

## 📋 Pipeline Breakdown

### 🎯 Stage 1: Customer Sentiment Analytics
* **Objective:** Quantify user satisfaction automatically across product tiers.
* **Process:** Ingests textual feedback, sanitizes encoding or formatting discrepancies, and executes batch predictions via a fine-tuned `DistilBERT` model. Text labels are dynamically mapped onto integer vectors (`predictions` and `true_labels`) to generate hard metrics like **Accuracy** and **F1-Score**.

### 🌍 Stage 2: Localization & Translation Quality Control
* **Objective:** Support expanding operations in Spain by translating English inputs while monitoring quality.
* **Process:** Isolates key contextual sentences and passes them through a `MarianMT` translation network. The generated `translated_review` is statistically cross-verified against human-certified ground truths via the Hugging Face `evaluate` framework, returning an explicit results dictionary containing exact **BLEU score** metrics.

### 🔍 Stage 3: Semantic Knowledge Extraction (Extractive QA)
* **Objective:** Enable the chatbot to query specific points of interest from long paragraphs instantaneously.
* **Process:** Feeds specific target questions alongside raw review contexts into a specialized `MiniLM` model trained on `SQuAD 2.0`. The architecture pinpoints exact character spans to deliver targeted, isolated answers (`answer`) without running generic keywords.

### 📝 Stage 4: Abstractive Feedback Condensation
* **Objective:** Shorten verbose, rambling customer stories into high-density briefings for corporate review.
* **Process:** Implements a heavy sequence-to-sequence `BART` model to read entire paragraphs and re-write them dynamically. The text generation engine is bounded by precise constraints (`min_length=50`, `max_length=55`) and set to deterministic decoding to guarantee consistent, highly accurate summaries (`summarized_text`).


In [50]:
# Import necessary packages
import pandas as pd
import torch

from transformers import logging
logging.set_verbosity(logging.WARNING)

## 1: Sentiment Analysis & Evaluation

In this task, we perform **Sentiment Analysis** on the car reviews dataset using a pre-trained Large Language Model (LLM) and evaluate its performance against the ground truth labels.

### 1. Data Loading & Preprocessing
* **File Reading:** The dataset is loaded from `data/car_reviews.csv` using a semicolon `;` separator and `utf-8-sig` encoding to handle hidden Byte Order Mark (BOM) characters.
* **Column Stripping:** `df.columns.str.strip()` is applied to remove any accidental whitespaces in column names (e.g., transforming `'Class '` to `'Class'`).

### 2. Model Inference
* **Pipeline Initialization:** We utilize Hugging Face's `pipeline` to load `distilbert-base-uncased-finetuned-sst-2-english`, a lightweight and accurate transformer model fine-tuned for sentiment classification.
* **Batch Prediction:** The `Review` column is converted into a Python list and passed to the classifier, storing raw dictionary outputs (labels and scores) in `predicted_labels`.

### 3. Label Mapping & Binary Conversion
To compute classification metrics, text-based categories are mapped into binary integers `{0, 1}`:
* **`predictions`**: Maps `POSITIVE` model outputs to `1` and `NEGATIVE` to `0`.
* **`true_labels`**: Converts the dataset's actual ground truth classes (`Class` column) into a binary format `{0, 1}` for direct alignment.

### 4. Metrics Evaluation
Using `scikit-learn`, we compute standard classification benchmarks to assess model reliability:
* **Accuracy (`accuracy_result`)**: Measures the global proportion of correct predictions.
* **F1-Score (`f1_result`)**: Calculates the harmonic mean of Precision and Recall, ensuring robust evaluation even if class distributions are imbalanced.

In [51]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score


df = pd.read_csv('data/car_reviews.csv', sep=';', encoding='utf-8-sig')

df.columns = df.columns.str.strip()

classifier = pipeline(task="sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

reviews = df['Review'].tolist()
predicted_labels = classifier(reviews)

predictions = [1 if label['label'] == 'POSITIVE' else 0 for label in predicted_labels]

true_labels = [1 if str(c).strip() == 'POSITIVE' else 0 for c in df['Class'].tolist()]

accuracy_result = accuracy_score(true_labels, predictions)
f1_result = f1_score(true_labels, predictions)

print(f"Accuracy: {accuracy_result:.2f}")
print(f"F1 Score: {f1_result:.2f}")

Device set to use cpu


Accuracy: 0.80
F1 Score: 0.86


## 2: Machine Translation & BLEU Score Evaluation

This component handles the localization of customer feedback for the Spanish market by translating English reviews into Spanish and evaluating the translation quality using the standard **BLEU (Bilingual Evaluation Understudy)** metric.

### 1. Text Extraction & Preprocessing
* **Target Selection:** The first review (`iloc[0]`) is selected from the dataset.
* **Sentence Tokenization:** The text is split by periods (`.`) to isolate the first two sentences. These sentences are then cleaned using `.strip()` and re-joined to form a concise input context for the translation model.

### 2. Machine Translation Pipeline
* **Model Inference:** We use Hugging Face's `pipeline` initialized with the pre-trained `Helsinki-NLP/opus-mt-en-es` sequence-to-sequence transformer model.
* **Output Generation:** The extracted English sentences are translated into Spanish, and the final generated string is stored in `translated_review`.

### 3. Metric Evaluation Setup (BLEU Score)
To systematically assess how close the model's translation is to human-quality text, we implement the BLEU metric:
* **Reference Loading:** Human-curated reference translations are read from `data/reference_translations.txt`. Each reference line is split into raw tokens.
* **Hugging Face Evaluate:** The modern `evaluate` library is utilized by loading the `"bleu"` metric instance.
* **Dictionary Computation:** By invoking `.compute()`, the library cross-references the model's `predictions` against the ground-truth `references`. 

### 4. Expected Output Structure
The `bleu_score` variable stores a comprehensive results dictionary containing:
* `bleu`: The cumulative BLEU score (scaled between 0.0 and 1.0).
* `precisions`: Individual n-gram precision scores (1-gram through 4-gram).
* `brevity_penalty`: A penalty factor applied if the model's generation is excessively short compared to the reference.
* `length_ratio` & lengths tracker for both translation and reference sets.

In [52]:
import nltk
from nltk.translate.bleu_score import sentence_bleu
import evaluate

translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

first_review = df['Review'].iloc[0]
sentences = first_review.split('.')
first_two_sentences = ". ".join(sentences[:2]).strip() + "."

translation_output = translator(first_two_sentences)
translated_review = translation_output[0]['translation_text']
bleu_metric = evaluate.load("bleu")
with open('data/reference_translations.txt', 'r', encoding='utf-8') as f:
    references = [line.strip().split() for line in f.readlines() if line.strip()]

candidate = translated_review.split()
bleu_score = bleu_metric.compute(
    predictions=[translated_review], 
    references=[references]
)

print(f"Translated Text: {translated_review}")
print("BLEU Score Dictionary:")
print(bleu_score)

Device set to use cpu


Translated Text: Estoy muy satisfecho con mi Nissan NV SL 2014. Uso esta camioneta para mis entregas de negocios y uso personal.
BLEU Score Dictionary:
{'bleu': 0.0, 'precisions': [0.09090909090909091, 0.0, 0.0, 0.0], 'brevity_penalty': 0.3849870989234837, 'length_ratio': 0.5116279069767442, 'translation_length': 22, 'reference_length': 43}


## 3: Extractive Question Answering (QA)

This task demonstrates the implementation of an **Extractive Question Answering** system to pinpoint specific customer insights from text without manual reading. Here, we analyze the second review to extract exact brand-related preferences.

### 1. Model Selection & Pipeline Setup
* **Model Choice:** We instantiate a Hugging Face `pipeline` using `deepset/minilm-uncased-squad2`. 
* **Architecture Advantage:** This model is based on the highly efficient **MiniLM** architecture, fine-tuned on the **SQuAD 2.0** (Stanford Question Answering Dataset). It is specially designed to extract precise answers from a given context and possesses the ability to recognize when a question cannot be answered based on the provided text.

### 2. Context & Question Formulation
* **`context`**: Captured dynamically from the second review in the dataset (`df['Review'].iloc[1]`), which highlights detailed brand aspects, vehicle reliability, and dealer experiences.
* **`question`**: A target query formulated as string: `"What did he like about the brand?"`.

### 3. Execution & Answer Extraction
* **Structured Input:** The model expects a Python dictionary (`qa_input`) containing both the target `question` and the reference `context`.
* **Inference:** The pipeline processes the text, locates the exact span of characters containing the highest probability answer, and stores the rich output metadata in `qa_result`.
* **`answer`**: The final actual text snippet is extracted using `qa_result['answer']`, isolating the specific attributes the customer appreciated (such as ride quality and reliability).

In [53]:
qa_predictor = pipeline("question-answering", model="deepset/minilm-uncased-squad2")

question = "What did he like about the brand?"
context = df['Review'].iloc[1]

qa_input = {'question': question, 'context': context}
qa_result = qa_predictor(qa_input)
answer = qa_result['answer']

print(f"Question: {question}")
print(f"Answer: {answer}\n")

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


Question: What did he like about the brand?
Answer: ride quality, reliability



## 4: Text Summarization

In this final task, we deploy an abstractive text summarization model to condense the longest and last review in our dataset into a concise, high-density summary while strictly satisfying token length constraints.

### 1. Model Selection & Pipeline Setup
* **Model Choice:** We initialize a Hugging Face `pipeline` utilizing **`facebook/bart-large-cnn`**.
* **Architecture Advantage:** This model leverages the **BART** (Bidirectional and Auto-Regressive Transformers) sequence-to-sequence architecture, heavily fine-tuned on the CNN/DailyMail dataset. It is highly optimized for generating coherent, human-like abstractive summaries rather than just extracting existing sentences.

### 2. Context Extraction & Parameter Tuning
* **Target Text (`last_review`)**: The model targets the final review in the dataset (`df['Review'].iloc[-1]`), which captures an extended narrative regarding an SUV purchase experience, financial arrangements, and driving impressions.
* **Hyperparameter Constraints:**
  * `max_length=55`: Strictly caps the generated summary output at a maximum of 55 tokens.
  * `min_length=50`: Forces the model to maintain depth, ensuring the summary does not drop below 50 tokens.
  * `do_sample=False`: Disables random sampling, enforcing **Deterministic Greedy/Beam Search** decoding to retrieve the most statistically probable and reliable summary sequence.

### 3. Execution & Storage
* **Inference:** The raw review is passed into the pipeline, and the model auto-regressively outputs the condensed text metadata inside a list of dictionaries (`summary_output`).
* **`summarized_text`**: The final, clean, and highly condensed text summary is extracted from `summary_output[0]['summary_text']`, fulfilling the CTO's requirement for a 50-55 token summary.

In [54]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

last_review = df['Review'].iloc[-1]

summary_output = summarizer(last_review, max_length=55, min_length=50, do_sample=False)
summarized_text = summary_output[0]['summary_text']

print(f"Summarized Text: {summarized_text}")

Device set to use cpu


Summarized Text: The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong
